Step 1 — Installing required packages (RapidFuzz + Pandas)

In [1]:
# Installing packages we need in Colab
# - rapidfuzz: fast fuzzy string matching
# - pandas: tables/dataframes for easy inspection

!pip -q install rapidfuzz pandas

import pandas as pd
from rapidfuzz import fuzz, process

print("Installed + imported: pandas + rapidfuzz")
print("pandas version:", pd.__version__)


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\manas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Installed + imported: pandas + rapidfuzz
pandas version: 2.2.3


Step 2 — Create folders + write SAMPLE B_out_001.json

In [2]:
# Creating /content/data folder and writing a SAMPLE B_out_001.json
# This is just for testing Module A until Module B is ready.

import os, json

# 1) Creating the folder (Colab uses /content as the working directory)
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# 2) Creating a sample input that matches Module B's exact schema
sample_B_out = {
    "case_id": "CASE_001",
    "excluded_mentions": [
        {
            "mention_id": "m1",
            "surface": "seizures",
            "negation_cue": "denies",
            "evidence": {"snippet": "Patient denies seizures.", "span": [0, 23]},
            "context": "patient"
        },
        {
            "mention_id": "m2",
            "surface": "ataxia",
            "negation_cue": "no",
            "evidence": {"snippet": "No ataxia.", "span": [25, 34]},
            "context": "patient"
        }
    ]
}

B_OUT_PATH = os.path.join(DATA_DIR, "B_out_001.json")

# 3) Saving the JSON file
with open(B_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sample_B_out, f, indent=2)

print("Created folder:", DATA_DIR)
print("Wrote sample file:", B_OUT_PATH)

# 4) Quick sanity check: reaing back and printing keys + number of mentions
with open(B_OUT_PATH, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("case_id:", loaded.get("case_id"))
print("excluded_mentions count:", len(loaded.get("excluded_mentions", [])))
print("mention_ids:", [m["mention_id"] for m in loaded["excluded_mentions"]])

Created folder: /content/data
Wrote sample file: /content/data\B_out_001.json
case_id: CASE_001
excluded_mentions count: 2
mention_ids: ['m1', 'm2']


Step 3 — Download the HPO ontology file (hp.obo)

In [3]:
# Downloading the official Human Phenotype Ontology file (hp.obo)

import urllib.request
import os

HPO_URL = "http://purl.obolibrary.org/obo/hp.obo"
HPO_PATH = "/content/data/hp.obo"

# Downloading the file only if it doesn't already exist
if not os.path.exists(HPO_PATH):
    print("Downloading HPO ontology...")
    urllib.request.urlretrieve(HPO_URL, HPO_PATH)
    print("Download complete.")
else:
    print("HPO file already exists.")

# Printing file info to confirm
file_size = os.path.getsize(HPO_PATH) / (1024 * 1024)

print("Saved to:", HPO_PATH)
print(f"File size: {file_size:.2f} MB")

# Showing first few lines so we confirm it's the ontology file
with open(HPO_PATH, "r", encoding="utf-8") as f:
    for _ in range(10):
        print(f.readline().strip())

HPO file already exists.
Saved to: /content/data/hp.obo
File size: 10.21 MB
format-version: 1.2
data-version: hp/releases/2026-02-16
subsetdef: hposlim_core "Core clinical terminology"
subsetdef: secondary_consequence "Consequence of a disorder in another organ system."
synonymtypedef: abbreviation "abbreviation"
synonymtypedef: allelic_requirement "allelic_requirement"
synonymtypedef: layperson "layperson term"
synonymtypedef: obsolete_synonym "discarded/obsoleted synonym"
synonymtypedef: plural_form "plural form"
synonymtypedef: uk_spelling "UK spelling"


Step 4 — Parsinng hp.obo into a DataFrame (id, label, synonyms list)

In [4]:
# Parsing hp.obo into a usable pandas DataFrame
# Output: terms_df with columns: id, label, synonyms (list of strings)
# we also collect is_a parents if present (not required for matching)

import pandas as pd
import re

def parse_hpo_obo(obo_path: str) -> pd.DataFrame:
    """
    Parse hp.obo and return a DataFrame with:
      - id: HPO ID (e.g., HP:0001250)
      - label: term name (e.g., Seizure)
      - synonyms: list of synonym strings (EXACT + RELATED + others if present)
      - parents: list of is_a parent IDs (optional; safe to ignore later)
    """
    terms = []
    current = None

    # Example synonym line:
    # synonym: "Seizures" EXACT [HPO:probinson]
    synonym_re = re.compile(r'^synonym:\s+"(.+?)"\s+(\w+)\s+\[.*\]')

    with open(obo_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Start of a new term
            if line == "[Term]":
                # Save previous term if valid
                if current and current.get("id") and current.get("label"):
                    terms.append(current)
                # Create a new term record
                current = {"id": None, "label": None, "synonyms": [], "parents": []}
                continue

            # If we haven't started a term yet, ignore header lines
            if current is None:
                continue

            # End of term block (sometimes there are blank lines)
            if line == "":
                continue

            # Parse fields inside a term
            if line.startswith("id:"):
                current["id"] = line.replace("id:", "").strip()

            elif line.startswith("name:"):
                current["label"] = line.replace("name:", "").strip()

            elif line.startswith("synonym:"):
                m = synonym_re.match(line)
                if m:
                    syn_text = m.group(1)      # text inside quotes
                    syn_scope = m.group(2)     # EXACT / RELATED / etc.
                    # We keep the synonym text (and you can also keep scope if you want)
                    current["synonyms"].append(syn_text)
                else:
                    # Fallback: try to extract quoted text even if format differs
                    quoted = re.findall(r'"(.*?)"', line)
                    if quoted:
                        current["synonyms"].append(quoted[0])

            elif line.startswith("is_a:"):
                # Example: is_a: HP:0000707 ! Abnormality of the nervous system
                parent_id = line.split("!")[0].replace("is_a:", "").strip()
                current["parents"].append(parent_id)

    # Save last term if file ended right after it
    if current and current.get("id") and current.get("label"):
        terms.append(current)

    return pd.DataFrame(terms)

# Running parser
HPO_PATH = "/content/data/hp.obo"
terms_df = parse_hpo_obo(HPO_PATH)

# Basic sanity checks + prints
print("Parsed hp.obo into DataFrame")
print("Rows (terms):", len(terms_df))
print("Columns:", list(terms_df.columns))

# Showing a few sample rows
display(terms_df.head(5))

# Showing one example term with synonyms (if available)
example_with_syn = terms_df[terms_df["synonyms"].map(len) > 0].head(1)
if len(example_with_syn) > 0:
    row = example_with_syn.iloc[0]
    print("\nExample term with synonyms:")
    print("ID:", row["id"])
    print("Label:", row["label"])
    print("First 5 synonyms:", row["synonyms"][:5])
else:
    print("\nNo synonyms found (unexpected) — tell me if you see this.")

Parsed hp.obo into DataFrame
Rows (terms): 19944
Columns: ['id', 'label', 'synonyms', 'parents']


,id,label,synonyms,parents
0,HP:0000001,All,[],[]
1,HP:0000002,Abnormality of body height,[Abnormality of body height],[HP:0001507]
2,HP:0000003,Multicystic kidney dysplasia,"[Multicystic dysplastic kidney, Multicystic ki...",[HP:0000107]
3,HP:0000005,Mode of inheritance,[Inheritance],[HP:0000001]
4,HP:0000006,Autosomal dominant inheritance,"[Autosomal dominant, Autosomal dominant form, ...",[HP:0034345]



Example term with synonyms:
ID: HP:0000002
Label: Abnormality of body height
First 5 synonyms: ['Abnormality of body height']


Step 5 — Normalization + Build Index (labels + synonyms)

In [5]:
# Building normalization + index for fast exact matching
# We will index BOTH:
# - labels (name)
# - synonyms
# so we can do Tier 1 exact match quickly.

import re
from collections import defaultdict

def normalize(text: str) -> str:
    """
    Normalize text for consistent matching:
    - lowercase
    - remove punctuation
    - collapse whitespace
    - trim spaces
    (optional) simple plural handling: seizures -> seizure, fevers -> fever
    """
    if text is None:
        return ""
    text = text.lower()

    # removing punctuation (keeping letters/numbers/spaces)
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # collapsing multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Only applying if word ends with 's' and length>3 (avoids "as", "is", etc.)
    # This is intentionally simple (not perfect English stemming).
    if text.endswith("s") and len(text) > 3:
        text_singular = text[:-1]
        # Using singular only if it doesn't create an empty string
        if len(text_singular) > 0:
            text = text_singular

    return text


def build_index(terms_df):
    """
    Build the structures required for matching:
    - surface_to_ids: normalized surface form -> set of HPO IDs that match it
    - id_to_label: HPO ID -> official label
    - all_surface_forms: list of all normalized surface forms (for fuzzy matching later)
    - surface_to_meta: normalized surface form -> list of (hpo_id, match_source)
        where match_source is either "label" or "synonym"
    """
    surface_to_ids = defaultdict(set)
    surface_to_meta = defaultdict(list)
    id_to_label = {}
    all_surface_forms = []

    for _, row in terms_df.iterrows():
        hpo_id = row["id"]
        label = row["label"]
        synonyms = row["synonyms"] if isinstance(row["synonyms"], list) else []

        # storing label lookup
        id_to_label[hpo_id] = label

        # index label
        n_label = normalize(label)
        if n_label:
            surface_to_ids[n_label].add(hpo_id)
            surface_to_meta[n_label].append((hpo_id, "label"))
            all_surface_forms.append(n_label)

        # index synonyms
        for syn in synonyms:
            n_syn = normalize(syn)
            if n_syn:
                surface_to_ids[n_syn].add(hpo_id)
                surface_to_meta[n_syn].append((hpo_id, "synonym"))
                all_surface_forms.append(n_syn)

    # de-duplicate surface forms (important for faster fuzzy matching later)
    all_surface_forms = sorted(set(all_surface_forms))

    return surface_to_ids, id_to_label, all_surface_forms, surface_to_meta


# Building index from the parsed terms_df
surface_to_ids, id_to_label, all_surface_forms, surface_to_meta = build_index(terms_df)

print("Index built!")
print("HPO terms loaded:", len(terms_df))
print("Unique normalized surface forms indexed (label+synonyms):", len(all_surface_forms))

# Quick test: showing how a known word normalizes
test_words = ["Seizures", "seizure.", "heart-murmur", "Fever!!", "  abnormality   of body height "]
print("\nNormalization sanity checks:")
for w in test_words:
    print(f"  '{w}'  ->  '{normalize(w)}'")

# Quick check: does the index contain normalized "seizure"?
print("\nDoes index contain 'seizure'? ->", "seizure" in surface_to_ids)
if "seizure" in surface_to_ids:
    example_ids = list(surface_to_ids["seizure"])[:5]
    print("Example HPO IDs for 'seizure':", example_ids)
    print("First example label:", id_to_label.get(example_ids[0]))

Index built!
HPO terms loaded: 19944
Unique normalized surface forms indexed (label+synonyms): 42439

Normalization sanity checks:
  'Seizures'  ->  'seizure'
  'seizure.'  ->  'seizure'
  'heart-murmur'  ->  'heart murmur'
  'Fever!!'  ->  'fever'
  '  abnormality   of body height '  ->  'abnormality of body height'

Does index contain 'seizure'? -> True
Example HPO IDs for 'seizure': ['HP:0001250']
First example label: Seizure


Step 6 — Tier 1 Exact Match Function (label_exact + synonym_exact)

In [6]:
# Tier 1 exact match function
# - exact match on normalized phrase vs normalized (label + synonyms)
# - returns candidates with required fields:
#   hpo_id, label, confidence, match_type

def exact_match_candidates(phrase: str, top_k: int = 3):
    """
    Tier 1: exact match on normalized phrase.
    Returns a list of candidate dicts (max top_k), each:
      {"hpo_id":..., "label":..., "confidence":..., "match_type":...}
    Confidence rules:
      - label_exact: 0.99
      - synonym_exact: 0.96
    """
    n_phrase = normalize(phrase)

    # If empty after normalization, return no candidates (caller will handle no_match)
    if not n_phrase:
        return []

    # Finding all matching HPO IDs (could be multiple)
    matched = surface_to_meta.get(n_phrase, [])  # list of (hpo_id, "label"/"synonym")
    if not matched:
        return []

    # Building candidate list with correct match_type + confidence
    candidates = []
    for hpo_id, source in matched:
        label = id_to_label.get(hpo_id)

        if source == "label":
            match_type = "label_exact"
            confidence = 0.99
        else:
            match_type = "synonym_exact"
            confidence = 0.96

        candidates.append({
            "hpo_id": hpo_id,
            "label": label,
            "confidence": confidence,
            "match_type": match_type
        })

    # Sorting: prefer label_exact over synonym_exact; then by confidence; then by ID
    priority = {"label_exact": 2, "synonym_exact": 1}
    candidates.sort(key=lambda x: (priority[x["match_type"]], x["confidence"], x["hpo_id"]), reverse=True)

    # Keeping only top_k
    return candidates[:top_k]


# ---- Quick tests on our sample phrases ----
test_phrases = ["seizures", "fever", "heart murmur", "heart-murmur", "randomnonsense"]
for p in test_phrases:
    cands = exact_match_candidates(p, top_k=3)
    print(f"\nPhrase: '{p}' (normalized -> '{normalize(p)}')")
    if cands:
        for c in cands:
            print("  ", c)
    else:
        print("   No exact match candidates")


Phrase: 'seizures' (normalized -> 'seizure')
   {'hpo_id': 'HP:0001250', 'label': 'Seizure', 'confidence': 0.99, 'match_type': 'label_exact'}
   {'hpo_id': 'HP:0001250', 'label': 'Seizure', 'confidence': 0.96, 'match_type': 'synonym_exact'}

Phrase: 'fever' (normalized -> 'fever')
   {'hpo_id': 'HP:0001945', 'label': 'Fever', 'confidence': 0.99, 'match_type': 'label_exact'}
   {'hpo_id': 'HP:0001945', 'label': 'Fever', 'confidence': 0.96, 'match_type': 'synonym_exact'}

Phrase: 'heart murmur' (normalized -> 'heart murmur')
   {'hpo_id': 'HP:0030148', 'label': 'Heart murmur', 'confidence': 0.99, 'match_type': 'label_exact'}
   {'hpo_id': 'HP:0030148', 'label': 'Heart murmur', 'confidence': 0.96, 'match_type': 'synonym_exact'}
   {'hpo_id': 'HP:0030148', 'label': 'Heart murmur', 'confidence': 0.96, 'match_type': 'synonym_exact'}

Phrase: 'heart-murmur' (normalized -> 'heart murmur')
   {'hpo_id': 'HP:0030148', 'label': 'Heart murmur', 'confidence': 0.99, 'match_type': 'label_exact'}
   

Step 7 — Tier 2 Fuzzy Match Function (RapidFuzz)

In [7]:
#  Tier 2 fuzzy matching using RapidFuzz
# - Uses normalized phrase vs all_surface_forms (normalized labels+synonyms)
# - Returns top_k candidates if score >= threshold
# Confidence rules:
#   fuzzy_match: score/100 capped at 0.95

from rapidfuzz import process, fuzz

def fuzzy_match_candidates(phrase: str, top_k: int = 3, threshold: int = 80):
    """
    Tier 2: fuzzy match using RapidFuzz over all_surface_forms.
    Returns list of candidate dicts (max top_k) with:
      {"hpo_id":..., "label":..., "confidence":..., "match_type":"fuzzy_match"}
    Only returns candidates where fuzzy score >= threshold.
    """
    n_phrase = normalize(phrase)
    if not n_phrase:
        return []

    # Get best matches from list of normalized surface forms
    # Each result = (matched_surface_form, score, index)
    results = process.extract(
        n_phrase,
        all_surface_forms,
        scorer=fuzz.WRatio,   # good general-purpose fuzzy scorer
        limit=top_k
    )

    candidates = []
    for matched_surface, score, _ in results:
        if score < threshold:
            continue

        # matched_surface maps to one or more HPO IDs
        for hpo_id in surface_to_ids.get(matched_surface, []):
            label = id_to_label.get(hpo_id)
            confidence = min(score / 100.0, 0.95)  # cap at 0.95

            candidates.append({
                "hpo_id": hpo_id,
                "label": label,
                "confidence": confidence,
                "match_type": "fuzzy_match"
            })

    # Sorting best confidence first, then stable by ID
    candidates.sort(key=lambda x: (x["confidence"], x["hpo_id"]), reverse=True)

    # Dedupe by hpo_id (keep best-scoring one)
    deduped = []
    seen = set()
    for c in candidates:
        if c["hpo_id"] in seen:
            continue
        seen.add(c["hpo_id"])
        deduped.append(c)

    return deduped[:top_k]


# ---- Quick tests for fuzzy matching ----
# These have small typos so exact match might fail, but fuzzy should succeed.
test_phrases_fuzzy = ["sezure", "fevr", "hart murmur", "high fevar", "randomnonsense"]
for p in test_phrases_fuzzy:
    cands = fuzzy_match_candidates(p, top_k=3, threshold=80)
    print(f"\nFuzzy test phrase: '{p}' (normalized -> '{normalize(p)}')")
    if cands:
        for c in cands:
            print("  ", c)
    else:
        print("   No fuzzy candidates above threshold")


Fuzzy test phrase: 'sezure' (normalized -> 'sezure')
   {'hpo_id': 'HP:0001250', 'label': 'Seizure', 'confidence': 0.923076923076923, 'match_type': 'fuzzy_match'}

Fuzzy test phrase: 'fevr' (normalized -> 'fevr')
   {'hpo_id': 'HP:0001945', 'label': 'Fever', 'confidence': 0.8888888888888888, 'match_type': 'fuzzy_match'}

Fuzzy test phrase: 'hart murmur' (normalized -> 'hart murmur')
   {'hpo_id': 'HP:0030148', 'label': 'Heart murmur', 'confidence': 0.95, 'match_type': 'fuzzy_match'}
   {'hpo_id': 'HP:0031670', 'label': 'Continuous heart murmur', 'confidence': 0.8571428571428571, 'match_type': 'fuzzy_match'}
   {'hpo_id': 'HP:0031668', 'label': 'Diastolic heart murmur', 'confidence': 0.8571428571428571, 'match_type': 'fuzzy_match'}

Fuzzy test phrase: 'high fevar' (normalized -> 'high fevar')
   {'hpo_id': 'HP:6000469', 'label': 'Elevated urine 2,3-dihydroxy-2-methylbutanoic acid level', 'confidence': 0.855, 'match_type': 'fuzzy_match'}
   {'hpo_id': 'HP:0034664', 'label': 'Elevated ur

Stepo 8 — match_phrase() wrapper (Exact → Fuzzy → No-match)

In [8]:
# match_phrase() wrapper
# Must return candidates list with required fields.
# Two-tier logic:
#   Tier 1: exact match
#   Tier 2: fuzzy match (with a simple guard to reduce nonsense matches)
#   Else: return exactly one no_match candidate

def _token_set(text: str):
    """Helper: get set of words (tokens) from normalized text."""
    if not text:
        return set()
    return set(text.split())

def match_phrase(phrase: str, top_k: int = 3, threshold: int = 80):
    """
    Two-tier matcher:
      1) exact_match_candidates
      2) fuzzy_match_candidates (with additional token-overlap guard)
      3) otherwise one no_match candidate

    Returns: list of candidates (length >=1 always, due to no_match fallback)
    """
    # --- Tier 1: exact ---
    exact = exact_match_candidates(phrase, top_k=top_k)

    if exact:
        # Dedupe by hpo_id (keep best match_type: label_exact > synonym_exact)
        priority = {"label_exact": 2, "synonym_exact": 1}
        best_by_id = {}
        for c in exact:
            hid = c["hpo_id"]
            if hid not in best_by_id:
                best_by_id[hid] = c
            else:
                # keep higher priority
                if priority[c["match_type"]] > priority[best_by_id[hid]["match_type"]]:
                    best_by_id[hid] = c

        # Return top_k in stable sorted order
        out = list(best_by_id.values())
        out.sort(key=lambda x: (priority[x["match_type"]], x["confidence"], x["hpo_id"]), reverse=True)
        return out[:top_k]

    # --- Tier 2: fuzzy ---
    n_phrase = normalize(phrase)
    phrase_tokens = _token_set(n_phrase)

    # We re-run extract here so we can apply token-overlap guard on the matched surface form
    results = process.extract(
        n_phrase,
        all_surface_forms,
        scorer=fuzz.WRatio,
        limit=top_k
    )

    fuzzy_candidates = []
    for matched_surface, score, _ in results:
        if score < threshold:
            continue

        matched_tokens = _token_set(matched_surface)

        # Simple guard: require at least ONE shared token
        # Prevents nonsense mappings like "randomnonsense" -> "Rheumatoid arthritis"
        if len(phrase_tokens.intersection(matched_tokens)) == 0:
            continue

        for hpo_id in surface_to_ids.get(matched_surface, []):
            fuzzy_candidates.append({
                "hpo_id": hpo_id,
                "label": id_to_label.get(hpo_id),
                "confidence": min(score / 100.0, 0.95),
                "match_type": "fuzzy_match"
            })

    # Dedupe fuzzy by hpo_id (keep best confidence)
    fuzzy_candidates.sort(key=lambda x: (x["confidence"], x["hpo_id"]), reverse=True)
    deduped = []
    seen = set()
    for c in fuzzy_candidates:
        if c["hpo_id"] in seen:
            continue
        seen.add(c["hpo_id"])
        deduped.append(c)

    if deduped:
        return deduped[:top_k]

    # --- No match fallback (EXACTLY ONE candidate) ---
    return [{
        "hpo_id": None,
        "label": None,
        "confidence": 0.0,
        "match_type": "no_match"
    }]


# ---- Quick tests for wrapper behavior ----
wrapper_tests = ["seizures", "sezure", "high fevar", "randomnonsense", "", "!!!"]
for p in wrapper_tests:
    cands = match_phrase(p, top_k=3, threshold=80)
    print(f"\nmatch_phrase('{p}') ->")
    for c in cands:
        print("  ", c)


match_phrase('seizures') ->
   {'hpo_id': 'HP:0001250', 'label': 'Seizure', 'confidence': 0.99, 'match_type': 'label_exact'}

match_phrase('sezure') ->
   {'hpo_id': None, 'label': None, 'confidence': 0.0, 'match_type': 'no_match'}

match_phrase('high fevar') ->
   {'hpo_id': 'HP:6000469', 'label': 'Elevated urine 2,3-dihydroxy-2-methylbutanoic acid level', 'confidence': 0.855, 'match_type': 'fuzzy_match'}
   {'hpo_id': 'HP:0034664', 'label': 'Elevated urine 2-hydroxy-3-methylvaleric acid level', 'confidence': 0.855, 'match_type': 'fuzzy_match'}
   {'hpo_id': 'HP:0034465', 'label': '2-hydroxyadipic aciduria', 'confidence': 0.855, 'match_type': 'fuzzy_match'}

match_phrase('randomnonsense') ->
   {'hpo_id': None, 'label': None, 'confidence': 0.0, 'match_type': 'no_match'}

match_phrase('') ->
   {'hpo_id': None, 'label': None, 'confidence': 0.0, 'match_type': 'no_match'}

match_phrase('!!!') ->
   {'hpo_id': None, 'label': None, 'confidence': 0.0, 'match_type': 'no_match'}


Step 9 — map_B_to_A() + save /content/data/A_out_001.json (with hpo_matches)

In [9]:
# Map B_out_001.json -> A_out_001.json using mention_id preserved exactly
# IMPORTANT: output schema uses "hpo_matches" (per your notice)

import json
import os

# ---- Patch match_phrase guard to allow 1-word typos like "sezure" -> "seizure" ----
def match_phrase(phrase: str, top_k: int = 3, threshold: int = 80):
    """
    Two-tier matcher:
      1) exact_match_candidates
      2) fuzzy match with safe guard
      3) else returns one no_match candidate

    Guard rule (deterministic):
      Accept fuzzy candidate if:
        - token overlap >= 1, OR
        - phrase is single-token AND matched_surface is single-token (typo-tolerant)
    """
    # --- Tier 1: exact ---
    exact = exact_match_candidates(phrase, top_k=top_k)
    if exact:
        priority = {"label_exact": 2, "synonym_exact": 1}
        best_by_id = {}
        for c in exact:
            hid = c["hpo_id"]
            if hid not in best_by_id:
                best_by_id[hid] = c
            else:
                if priority[c["match_type"]] > priority[best_by_id[hid]["match_type"]]:
                    best_by_id[hid] = c
        out = list(best_by_id.values())
        out.sort(key=lambda x: (priority[x["match_type"]], x["confidence"], x["hpo_id"]), reverse=True)
        return out[:top_k]

    # --- Tier 2: fuzzy ---
    n_phrase = normalize(phrase)
    if not n_phrase:
        return [{
            "hpo_id": None,
            "label": None,
            "confidence": 0.0,
            "match_type": "no_match"
        }]

    phrase_tokens = set(n_phrase.split())

    results = process.extract(
        n_phrase,
        all_surface_forms,
        scorer=fuzz.WRatio,
        limit=top_k
    )

    fuzzy_candidates = []
    for matched_surface, score, _ in results:
        if score < threshold:
            continue

        matched_tokens = set(matched_surface.split())

        token_overlap = len(phrase_tokens.intersection(matched_tokens))

        # Guard: allow overlap OR single-token typo case
        if token_overlap == 0:
            if not (len(phrase_tokens) == 1 and len(matched_tokens) == 1):
                continue

        for hpo_id in surface_to_ids.get(matched_surface, []):
            fuzzy_candidates.append({
                "hpo_id": hpo_id,
                "label": id_to_label.get(hpo_id),
                "confidence": min(score / 100.0, 0.95),
                "match_type": "fuzzy_match"
            })

    # Dedupe by hpo_id
    fuzzy_candidates.sort(key=lambda x: (x["confidence"], x["hpo_id"]), reverse=True)
    deduped = []
    seen = set()
    for c in fuzzy_candidates:
        if c["hpo_id"] in seen:
            continue
        seen.add(c["hpo_id"])
        deduped.append(c)

    if deduped:
        return deduped[:top_k]

    # --- No match fallback (EXACTLY ONE) ---
    return [{
        "hpo_id": None,
        "label": None,
        "confidence": 0.0,
        "match_type": "no_match"
    }]


def map_B_to_A(B_out_path: str, A_out_path: str, top_k: int = 3, threshold: int = 80):
    """
    Reads B_out JSON (excluded_mentions) and writes A_out JSON (mapped_exclusions).
    CRITICAL: mention_id preserved EXACTLY.
    Output schema (your required version):
      {
        "case_id": "...",
        "mapped_exclusions": [
          {"mention_id": "...", "surface": "...", "hpo_matches": [ ...candidates... ]}
        ]
      }
    """
    with open(B_out_path, "r", encoding="utf-8") as f:
        b = json.load(f)

    case_id = b.get("case_id")
    excluded = b.get("excluded_mentions", [])

    mapped_exclusions = []
    for m in excluded:
        mention_id = m["mention_id"]   # MUST NOT CHANGE
        surface = m["surface"]

        candidates = match_phrase(surface, top_k=top_k, threshold=threshold)

        mapped_exclusions.append({
            "mention_id": mention_id,
            "surface": surface,
            "hpo_matches": candidates
        })

    a_out = {
        "case_id": case_id,
        "mapped_exclusions": mapped_exclusions
    }

    with open(A_out_path, "w", encoding="utf-8") as f:
        json.dump(a_out, f, indent=2)

    return a_out


# Run mapping on our sample file
B_OUT_PATH = "/content/data/B_out_001.json"
A_OUT_PATH = "/content/data/A_out_001.json"

a_out_obj = map_B_to_A(B_OUT_PATH, A_OUT_PATH, top_k=3, threshold=80)

print(" Wrote A_out file to:", A_OUT_PATH)
print("case_id:", a_out_obj["case_id"])
print("mapped_exclusions count:", len(a_out_obj["mapped_exclusions"]))

# Quick check: mention_ids preserved
print("mention_ids preserved:", [x["mention_id"] for x in a_out_obj["mapped_exclusions"]])

# Quick sanity test: typo now should fuzzy-match
print("\nSanity check typo 'sezure' now:")
print(match_phrase("sezure", top_k=3, threshold=80))

 Wrote A_out file to: /content/data/A_out_001.json
case_id: CASE_001
mapped_exclusions count: 2
mention_ids preserved: ['m1', 'm2']

Sanity check typo 'sezure' now:
[{'hpo_id': 'HP:0001250', 'label': 'Seizure', 'confidence': 0.923076923076923, 'match_type': 'fuzzy_match'}]


Step 10 — Debug table + show saved A_out_001.json contents

In [10]:
# Debug prints + show saved file contents
# Requirements:
# - print a debug table showing surface → chosen top candidate
# - show how many HPO terms loaded and how many surface forms indexed
# - show saved A_out_001.json contents

import json
import pandas as pd

A_OUT_PATH = "/content/data/A_out_001.json"

# 1) Loading A_out file
with open(A_OUT_PATH, "r", encoding="utf-8") as f:
    a_loaded = json.load(f)

print("Loaded:", A_OUT_PATH)
print("case_id:", a_loaded["case_id"])
print("mapped_exclusions count:", len(a_loaded["mapped_exclusions"]))

# 2) Showing counts required
print("\n--- HPO Load + Index Stats ---")
print("HPO terms loaded:", len(terms_df))
print("Unique normalized surface forms indexed (label+synonyms):", len(all_surface_forms))

# 3) Building debug table: surface -> TOP candidate only
rows = []
for item in a_loaded["mapped_exclusions"]:
    mention_id = item["mention_id"]
    surface = item["surface"]
    matches = item["hpo_matches"]

    top = matches[0] if matches else {"hpo_id": None, "label": None, "confidence": 0.0, "match_type": "no_match"}

    rows.append({
        "mention_id": mention_id,
        "surface": surface,
        "top_hpo_id": top["hpo_id"],
        "top_label": top["label"],
        "top_confidence": top["confidence"],
        "top_match_type": top["match_type"]
    })

debug_df = pd.DataFrame(rows)

print("\n--- Debug Table (surface -> top candidate) ---")
display(debug_df)

# 4) Printing the entire saved JSON so you can confirm schema
print("\n--- A_out_001.json (full contents) ---")
print(json.dumps(a_loaded, indent=2))

Loaded: /content/data/A_out_001.json
case_id: CASE_001
mapped_exclusions count: 2

--- HPO Load + Index Stats ---
HPO terms loaded: 19944
Unique normalized surface forms indexed (label+synonyms): 42439

--- Debug Table (surface -> top candidate) ---


,mention_id,surface,top_hpo_id,top_label,top_confidence,top_match_type
0,m1,seizures,HP:0001250,Seizure,0.99,label_exact
1,m2,ataxia,HP:0001251,Ataxia,0.99,label_exact



--- A_out_001.json (full contents) ---
{
  "case_id": "CASE_001",
  "mapped_exclusions": [
    {
      "mention_id": "m1",
      "surface": "seizures",
      "hpo_matches": [
        {
          "hpo_id": "HP:0001250",
          "label": "Seizure",
          "confidence": 0.99,
          "match_type": "label_exact"
        }
      ]
    },
    {
      "mention_id": "m2",
      "surface": "ataxia",
      "hpo_matches": [
        {
          "hpo_id": "HP:0001251",
          "label": "Ataxia",
          "confidence": 0.99,
          "match_type": "label_exact"
        }
      ]
    }
  ]
}


A — Cache the parsed HPO terms (terms_df)

In [11]:
# Cache parsed HPO terms so future runs are faster
import os
import pandas as pd

TERMS_CACHE_PATH = "/content/data/hpo_terms_df.pkl"

terms_df.to_pickle(TERMS_CACHE_PATH)

print("Cached terms_df to:", TERMS_CACHE_PATH)
print("Cached rows (terms):", len(terms_df))

Cached terms_df to: /content/data/hpo_terms_df.pkl
Cached rows (terms): 19944


Cell B — Cache the built index (lookup structures)

In [12]:
# Cache index objects for fast reruns
import pickle

INDEX_CACHE_PATH = "/content/data/hpo_index.pkl"

index_bundle = {
    "surface_to_ids": dict(surface_to_ids),   # convert defaultdict -> dict
    "id_to_label": dict(id_to_label),
    "all_surface_forms": list(all_surface_forms),
    "surface_to_meta": dict(surface_to_meta)
}

with open(INDEX_CACHE_PATH, "wb") as f:
    pickle.dump(index_bundle, f)

print("Cached index bundle to:", INDEX_CACHE_PATH)
print("Surface forms cached:", len(index_bundle["all_surface_forms"]))

Cached index bundle to: /content/data/hpo_index.pkl
Surface forms cached: 42439


Test Cell — Create B_out from this note + run Module A

In [13]:
import json, os

DATA_DIR = "/content/data"
B_OUT_PATH = os.path.join(DATA_DIR, "B_out_001.json")
A_OUT_PATH = os.path.join(DATA_DIR, "A_out_001.json")

# This mimics what Person B would output: ONLY excluded/negated mentions
sample_B_out = {
  "case_id": "CASE_001",
  "excluded_mentions": [
    {
      "mention_id": "m1",
      "surface": "seizures",
      "negation_cue": "no history of",
      "evidence": {"snippet": "No history of seizures", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m2",
      "surface": "severe headaches",
      "negation_cue": "no history of",
      "evidence": {"snippet": "No history of ... severe headaches", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m3",
      "surface": "visual hallucinations",
      "negation_cue": "no history of",
      "evidence": {"snippet": "No history of ... visual hallucinations", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m4",
      "surface": "chest pain",
      "negation_cue": "denies",
      "evidence": {"snippet": "Denies chest pain", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m5",
      "surface": "palpitations",
      "negation_cue": "denies",
      "evidence": {"snippet": "Denies ... palpitations", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m6",
      "surface": "shortness of breath",
      "negation_cue": "denies",
      "evidence": {"snippet": "Denies ... shortness of breath", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m7",
      "surface": "syncopal episodes",
      "negation_cue": "denies",
      "evidence": {"snippet": "Denies ... syncopal episodes", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m8",
      "surface": "REM sleep behavior disorder",
      "negation_cue": "no",
      "evidence": {"snippet": "no reports of REM sleep behavior disorder", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m9",
      "surface": "spasticity",
      "negation_cue": "no evidence of",
      "evidence": {"snippet": "no evidence of spasticity", "span": [0, 0]},
      "context": "patient"
    },
    {
      "mention_id": "m10",
      "surface": "rigidity",
      "negation_cue": "no evidence of",
      "evidence": {"snippet": "no evidence of ... rigidity", "span": [0, 0]},
      "context": "patient"
    }
  ]
}

# Write B_out
with open(B_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sample_B_out, f, indent=2)

print("✅ Wrote test B_out:", B_OUT_PATH)
print("Excluded mentions:", len(sample_B_out["excluded_mentions"]))

# Run your Module A mapper
a_out = map_B_to_A(B_OUT_PATH, A_OUT_PATH, top_k=3, threshold=80)

print("✅ Wrote A_out:", A_OUT_PATH)
print("Mapped exclusions:", len(a_out["mapped_exclusions"]))

# Print debug lines: surface -> top mapping
for item in a_out["mapped_exclusions"]:
    top = item["hpo_matches"][0]
    print(f"{item['mention_id']} | {item['surface']} -> {top['hpo_id']} | {top['label']} | {top['confidence']:.2f} | {top['match_type']}")

✅ Wrote test B_out: /content/data\B_out_001.json
Excluded mentions: 10
✅ Wrote A_out: /content/data\A_out_001.json
Mapped exclusions: 10
m1 | seizures -> HP:0001250 | Seizure | 0.99 | label_exact
m2 | severe headaches -> HP:0012828 | Severe | 0.90 | fuzzy_match
m3 | visual hallucinations -> HP:0002367 | Visual hallucination | 0.99 | label_exact
m4 | chest pain -> HP:0100749 | Chest pain | 0.99 | label_exact
m5 | palpitations -> HP:0001962 | Palpitations | 0.99 | label_exact
m6 | shortness of breath -> HP:0002094 | Dyspnea | 0.96 | synonym_exact
m7 | syncopal episodes -> HP:0500261 | Triggered by anesthetics | 0.85 | fuzzy_match
m8 | REM sleep behavior disorder -> HP:5200291 | REM sleep behavior disorder | 0.99 | label_exact
m9 | spasticity -> HP:0001257 | Spasticity | 0.99 | label_exact
m10 | rigidity -> HP:0002063 | Rigidity | 0.99 | label_exact


Patch normalization + patch match_phrase

In [14]:
import re
from rapidfuzz import process, fuzz

# words that shouldn't influence phenotype mapping
MODIFIER_WORDS = {
    "severe","mild","moderate","occasional","intermittent","progressive","acute","chronic",
    "major","minor","significant","notable","generally","recent","slowly"
}

GENERIC_WORDS = {
    "history","evidence","episode","episodes","involvement","symptom","symptoms","complaint","complaints"
}

def normalize(text: str) -> str:
    """
    Improved normalization:
    - lowercase
    - remove punctuation
    - collapse whitespace
    - safer plural handling
    """
    if text is None:
        return ""

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    if not tokens:
        return ""

    last = tokens[-1]

    # safer plural handling
    if last.endswith(("ches","shes","xes","zes","ses")):
        last = last[:-1]          # headaches -> headache
    elif last.endswith("es") and len(last) > 4:
        last = last[:-1]          # episodes -> episode
    elif last.endswith("s") and len(last) > 3:
        last = last[:-1]          # seizures -> seizure

    tokens[-1] = last
    return " ".join(tokens)


def normalize_for_matching_input(phrase: str) -> str:
    n = normalize(phrase)
    if not n:
        return ""

    # clinical rewrite: "syncopal" adjective -> "syncope" noun
    n = n.replace("syncopal", "syncope")

    tokens = [
        t for t in n.split()
        if t not in MODIFIER_WORDS and t not in GENERIC_WORDS
    ]
    return " ".join(tokens).strip()


# patched exact matcher
def exact_match_candidates(phrase: str, top_k: int = 3):

    n_phrase = normalize_for_matching_input(phrase)

    if not n_phrase:
        return []

    matched = surface_to_meta.get(n_phrase, [])

    if not matched:
        return []

    candidates = []

    for hpo_id, source in matched:

        label = id_to_label.get(hpo_id)

        if source == "label":
            match_type = "label_exact"
            confidence = 0.99
        else:
            match_type = "synonym_exact"
            confidence = 0.96

        candidates.append({
            "hpo_id": hpo_id,
            "label": label,
            "confidence": confidence,
            "match_type": match_type
        })

    # prefer label_exact
    priority = {"label_exact": 2, "synonym_exact": 1}

    best_by_id = {}

    for c in candidates:
        hid = c["hpo_id"]
        if hid not in best_by_id or priority[c["match_type"]] > priority[best_by_id[hid]["match_type"]]:
            best_by_id[hid] = c

    out = list(best_by_id.values())

    out.sort(
        key=lambda x: (priority[x["match_type"]], x["confidence"], x["hpo_id"]),
        reverse=True
    )

    return out[:top_k]


# patched matcher
def match_phrase(phrase: str, top_k: int = 3, threshold: int = 80):

    # exact tier
    exact = exact_match_candidates(phrase, top_k=top_k)

    if exact:
        return exact

    # fuzzy tier
    n_phrase = normalize_for_matching_input(phrase)

    if not n_phrase:
        return [{
            "hpo_id": None,
            "label": None,
            "confidence": 0.0,
            "match_type": "no_match"
        }]

    phrase_tokens = set(n_phrase.split())

    one_word = len(phrase_tokens) == 1

    scorer = fuzz.WRatio if one_word else fuzz.token_sort_ratio
    eff_threshold = threshold if one_word else max(threshold, 85)

    results = process.extract(
        n_phrase,
        all_surface_forms,
        scorer=scorer,
        limit=top_k
    )

    fuzzy_candidates = []

    for matched_surface, score, _ in results:

        if score < eff_threshold:
            continue

        matched_tokens = set(matched_surface.split())

        token_overlap = len(phrase_tokens.intersection(matched_tokens))

        if token_overlap == 0:
            if not (one_word and len(matched_tokens) == 1):
                continue

        for hpo_id in surface_to_ids.get(matched_surface, []):

            fuzzy_candidates.append({
                "hpo_id": hpo_id,
                "label": id_to_label.get(hpo_id),
                "confidence": min(score / 100.0, 0.95),
                "match_type": "fuzzy_match"
            })

    fuzzy_candidates.sort(
        key=lambda x: (x["confidence"], x["hpo_id"]),
        reverse=True
    )

    deduped = []
    seen = set()

    for c in fuzzy_candidates:
        if c["hpo_id"] in seen:
            continue
        seen.add(c["hpo_id"])
        deduped.append(c)

    if deduped:
        return deduped[:top_k]

    return [{
        "hpo_id": None,
        "label": None,
        "confidence": 0.0,
        "match_type": "no_match"
    }]


print("✅ Matching logic patched successfully")
print("severe headaches ->", normalize_for_matching_input("severe headaches"))
print("syncopal episodes ->", normalize_for_matching_input("syncopal episodes"))

✅ Matching logic patched successfully
severe headaches -> headache
syncopal episodes -> syncope


In [15]:
print("/content/data/A_out_001.json")

/content/data/A_out_001.json


In [16]:
import json

with open("/content/data/A_out_001.json") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "case_id": "CASE_001",
  "mapped_exclusions": [
    {
      "mention_id": "m1",
      "surface": "seizures",
      "hpo_matches": [
        {
          "hpo_id": "HP:0001250",
          "label": "Seizure",
          "confidence": 0.99,
          "match_type": "label_exact"
        }
      ]
    },
    {
      "mention_id": "m2",
      "surface": "severe headaches",
      "hpo_matches": [
        {
          "hpo_id": "HP:0012828",
          "label": "Severe",
          "confidence": 0.9,
          "match_type": "fuzzy_match"
        },
        {
          "hpo_id": "HP:0002315",
          "label": "Headache",
          "confidence": 0.9,
          "match_type": "fuzzy_match"
        },
        {
          "hpo_id": "HP:0033181",
          "label": "Spinal epidural abscess",
          "confidence": 0.855,
          "match_type": "fuzzy_match"
        }
      ]
    },
    {
      "mention_id": "m3",
      "surface": "visual hallucinations",
      "hpo_matches": [
        {
          

Run Module A on B’s real file

In [19]:
import json, os
import pandas as pd

B_IN_PATH = "../data/moduleB_output.json"
OUT_DIR = "../data"

with open(B_IN_PATH, "r", encoding="utf-8") as f:
    b_data = json.load(f)

print("Loaded type:", type(b_data))
print("Number of cases:", len(b_data))

all_debug_rows = []
a_outputs = []

for case_obj in b_data:

    case_id = case_obj["case_id"]

    # TEMP FILE REQUIRED BY map_B_to_A()
    temp_B_path = os.path.join(OUT_DIR, f"temp_B_{case_id}.json")

    with open(temp_B_path, "w") as f:
        json.dump(case_obj, f, indent=2)

    # Dummy A output path (we ignore this)
    dummy_A_path = os.path.join(OUT_DIR, "A_dummy.json")

    # Run mapping
    a_out = map_B_to_A(temp_B_path, dummy_A_path, top_k=3, threshold=80)
    a_outputs.append(a_out)

    # DELETE TEMP FILE
    if os.path.exists(temp_B_path):
        os.remove(temp_B_path)

    # Debug rows
    for item in a_out["mapped_exclusions"]:
        top = item["hpo_matches"][0]

        all_debug_rows.append({
            "case_id": case_id,
            "mention_id": item["mention_id"],
            "surface": item["surface"],
            "top_hpo_id": top["hpo_id"],
            "top_label": top["label"],
            "confidence": top["confidence"],
            "match_type": top["match_type"]
        })

print("✅ Processed all cases with Module A")

# Show small debug table
debug_df = pd.DataFrame(all_debug_rows)
display(debug_df.head(20))

# Save combined output
combined_out = os.path.join(OUT_DIR, "moduleA_output.json")

with open(combined_out, "w") as f:
    json.dump(a_outputs, f, indent=2)

print("✅ Final combined file saved:", combined_out)

Loaded type: <class 'list'>
Number of cases: 10
✅ Processed all cases with Module A


,case_id,mention_id,surface,top_hpo_id,top_label,confidence,match_type
0,CASE_001,m1,loss of consciousness,HP:0007185,Loss of consciousness,0.99,label_exact
1,CASE_001,m2,seizures,HP:0001250,Seizure,0.99,label_exact
2,CASE_001,m3,headaches,HP:0002315,Headache,0.99,label_exact
3,CASE_001,m4,visual hallucinations,HP:0002367,Visual hallucination,0.99,label_exact
4,CASE_001,m5,palpitations,HP:0001962,Palpitations,0.99,label_exact
5,CASE_001,m6,shortness of breath,HP:0002094,Dyspnea,0.96,synonym_exact
6,CASE_001,m7,syncopal episodes,HP:0001279,Syncope,0.99,label_exact
7,CASE_001,m8,REM sleep behavior disorder,HP:5200291,REM sleep behavior disorder,0.99,label_exact
8,CASE_001,m9,weight loss,HP:0001824,Weight loss,0.99,label_exact
9,CASE_001,m10,spasticity,HP:0001257,Spasticity,0.99,label_exact


✅ Final combined file saved: ../data\moduleA_output.json


In [18]:
from pathlib import Path
import json

# Directory setup
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "moduleA_output.json"

combined_output = []

print("Building combined Module A output...")

for case in b_data:
    case_id = case["case_id"]
    excluded_list = case["excluded_mentions"]

    mapped_mentions = []

    for mention in excluded_list:
        surface = mention.get("surface", "").strip()
        norm_surface = surface.lower().strip()

        # ---------------------------------------
        # Step 1: Exact match using your hpo_dict
        # ---------------------------------------
        if norm_surface in hpo_dict:
            hpo_id = hpo_dict[norm_surface]

        else:
            # ---------------------------------------
            # Step 2: Fuzzy match using your function
            # ---------------------------------------
            hpo_id, score = get_best_match(norm_surface)

            # You can apply a threshold if you want
            if score < 80:
                hpo_id = None

        mapped_mentions.append({
            "surface": surface,
            "hpo_id": hpo_id
        })

    combined_output.append({
        "case_id": case_id,
        "excluded": mapped_mentions
    })

# Save a SINGLE combined output JSON
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(combined_output, f, indent=2, ensure_ascii=False)

print("Saved combined Module A output to:", OUTPUT_PATH)
print("Total cases:", len(combined_output))

Building combined Module A output...


NameError: name 'hpo_dict' is not defined